In [ ]:
import torch
import torchvision.models as models
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2


training_data = datasets.FashionMNIST(
    root = "data" ,
    train = True,
    download= True,
    transform= v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32, scale= True)])
)
testing_data = datasets.FashionMNIST(
    root= 'data', 
    train = True ,
    download= True,
    transform= v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32,scale= True)])
)

train_dataloader = DataLoader(training_data, batch_size= 64)
test_dataloader = DataLoader(testing_data, batch_size= 64)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [ ]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)


In [ ]:
def train_loop(dataloader,model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred , y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss , current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss , correct = 0 , 0

    with torch.no_grad():
        for X , y in dataloader:
            pred = model(X)
            test_loss  += loss_fn(pred,y).item()
            correct += (pred.argmax(1)== y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracu: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr= learning_rate)

epochs = 10

for t in range(epochs):
    print(f"EPOCH {t+1}\n---------------------------------------------")
    train_loop(test_dataloader , model , loss_fn , optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")



In [ ]:
# saving the model 
model = models.vgg16(weights = 'IMAGENET1K_V1')
torch.save(model.state_dict(), 'model_weights.pth')


In [ ]:
model = models.vgg16()
model.load_state_dict(torch.load('model_weights.pth', weights_only= True))
model.eval()

torch.save(model, 'model.pth')